# 第13章: PyTorch の仕組みを最新環境で検証する

この Notebook は、原本 `machine-learning-book/ch13/` の内容を、`uv` 管理下の最新依存関係で継続検証しやすい形に再構成したものです。原本の教育的な意図を保ちつつ、CI で不安定になりやすいネットワーク依存のデータ取得や未導入の追加フレームワーク依存は避け、純粋な PyTorch と scikit-learn だけで章の主要トピックを確認します。

## この Notebook で扱う内容

- 計算グラフと自動微分の基本を、最小限のテンソル演算で確認する。
- `nn.Sequential` と `nn.Module` を用いた XOR 分類を比較し、非線形モデルの必要性を見る。
- カスタムレイヤーの例として入力ノイズ付き線形層を実装する。
- 原本の回帰・画像分類プロジェクトは、CI で再現しやすいローカル同梱データセットに置き換えて検証する。
- 原本の PyTorch Lightning / Ignite の節は、この検証リポジトリの現行依存には含まれていないため、純粋な PyTorch の範囲に絞って再構成する。

In [ ]:
from importlib.metadata import version
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Image, display
from sklearn.datasets import load_diabetes, load_digits
from sklearn.metrics import accuracy_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = next(
    (candidate.resolve() for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'machine-learning-book').exists()),
    None,
)
assert PROJECT_ROOT is not None, 'machine-learning-book/ を含むプロジェクトルートが見つかりません。'

FIG_DIR = PROJECT_ROOT / 'machine-learning-book' / 'ch13' / 'figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'torch', 'nbformat']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)

SEED = 1
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Project root: {PROJECT_ROOT}')
print(f'Matplotlib backend: {matplotlib.get_backend()}')
package_versions

## 原本図版の参照

移行版 Notebook は `src/ch13/` に配置しますが、図版は読み取り専用の `machine-learning-book/ch13/figures/` をそのまま参照します。これにより、原本サブモジュールを編集せずに章の文脈を保てます。

In [ ]:
display(Image(filename=str(FIG_DIR / '13_01.png'), width=420))
display(Image(filename=str(FIG_DIR / '13_07.png'), width=700))

## 計算グラフと自動微分

原本前半の要点は、PyTorch がテンソル演算の履歴を追跡し、`backward()` により損失から勾配を自動計算できる点です。まずはごく小さな式で挙動を確認します。

In [ ]:
def compute_z(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    return torch.add(torch.mul(torch.sub(a, b), 2), c)

print('Scalar inputs:', compute_z(torch.tensor(1), torch.tensor(2), torch.tensor(3)))
print('Rank-1 inputs:', compute_z(torch.tensor([1]), torch.tensor([2]), torch.tensor([3])))

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
x = torch.tensor([1.4])
y = torch.tensor([2.1])

pred = w * x + b
loss = torch.sum((y - pred) ** 2)
loss.backward()

print(f'Prediction: {pred.item():.3f}')
print(f'Loss: {loss.item():.3f}')
print(f'dL/dw: {w.grad.item():.3f}')
print(f'dL/db: {b.grad.item():.3f}')
print('解析的な dL/dw:', (2 * x * ((w * x + b) - y)).item())

## XOR 分類で `nn.Sequential` と `nn.Module` を比較する

XOR は線形分離できないため、単層モデルと多層モデルの差が分かりやすく出ます。原本の流れに沿って、まず単純な線形分類器を試し、その後に隠れ層を持つモデルへ切り替えます。

In [ ]:
torch.manual_seed(SEED)

x_np = np.random.uniform(-1.0, 1.0, size=(200, 2)).astype(np.float32)
y_np = (x_np[:, 0] * x_np[:, 1] >= 0).astype(np.float32)

x_train_np, x_valid_np, y_train_np, y_valid_np = train_test_split(
    x_np, y_np, test_size=0.5, random_state=SEED, stratify=y_np
)

x_train = torch.tensor(x_train_np)
y_train = torch.tensor(y_train_np)
x_valid = torch.tensor(x_valid_np)
y_valid = torch.tensor(y_valid_np)

train_loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=16,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)


def train_binary_classifier(model: nn.Module, train_dl: DataLoader, x_valid: torch.Tensor, y_valid: torch.Tensor, *, epochs: int, lr: float):
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}

    for _ in range(epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0.0
        total_examples = 0

        for x_batch, y_batch in train_dl:
            optimizer.zero_grad()
            pred = model(x_batch).squeeze(1)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(x_batch)
            running_correct += ((pred >= 0.5).float() == y_batch).sum().item()
            total_examples += len(x_batch)

        model.eval()
        with torch.no_grad():
            valid_pred = model(x_valid).squeeze(1)
            valid_loss = loss_fn(valid_pred, y_valid).item()
            valid_acc = ((valid_pred >= 0.5).float() == y_valid).float().mean().item()

        history['train_loss'].append(running_loss / total_examples)
        history['valid_loss'].append(valid_loss)
        history['train_acc'].append(running_correct / total_examples)
        history['valid_acc'].append(valid_acc)

    return history


def plot_xor_results(ax, model: nn.Module, title: str):
    grid_x1, grid_x2 = np.meshgrid(np.linspace(-1.1, 1.1, 200), np.linspace(-1.1, 1.1, 200))
    grid = np.column_stack([grid_x1.ravel(), grid_x2.ravel()]).astype(np.float32)
    with torch.no_grad():
        surface = model(torch.tensor(grid)).reshape(grid_x1.shape).numpy()

    ax.contourf(grid_x1, grid_x2, surface, levels=np.linspace(0, 1, 11), cmap='RdYlBu', alpha=0.65)
    ax.scatter(x_valid_np[:, 0], x_valid_np[:, 1], c=y_valid_np, cmap='coolwarm', edgecolor='black', s=35)
    ax.set_title(title)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

linear_model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
linear_history = train_binary_classifier(linear_model, train_loader, x_valid, y_valid, epochs=120, lr=0.05)

mlp_model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid(),
)
mlp_history = train_binary_classifier(mlp_model, train_loader, x_valid, y_valid, epochs=120, lr=0.1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(linear_history['valid_acc'], label='linear')
axes[0].plot(mlp_history['valid_acc'], label='mlp')
axes[0].set_title('Validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(alpha=0.3)
axes[0].legend()

plot_xor_results(axes[1], linear_model, 'Linear decision surface')
plot_xor_results(axes[2], mlp_model, 'MLP decision surface')
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"線形モデルの最終検証精度: {linear_history['valid_acc'][-1]:.3f}")
print(f"多層モデルの最終検証精度: {mlp_history['valid_acc'][-1]:.3f}")

class FlexibleXORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid(),
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

    def predict(self, x: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            probs = self(torch.tensor(x, dtype=torch.float32)).squeeze(1)
        return (probs >= 0.5).to(torch.int64).numpy()

module_model = FlexibleXORNet()
module_history = train_binary_classifier(module_model, train_loader, x_valid, y_valid, epochs=120, lr=0.1)
module_predictions = module_model.predict(x_valid_np[:8])

print(f"FlexibleXORNet の最終検証精度: {module_history['valid_acc'][-1]:.3f}")
print('先頭 8 サンプルの予測:', module_predictions.tolist())

## カスタムレイヤーを実装する

原本の `NoisyLinear` と同様に、学習時だけ入力へガウスノイズを加える層を実装します。`nn.Parameter` を明示的に持つことで、通常のレイヤーと同じように最適化対象へ組み込めます。

In [ ]:
class NoisyLinear(nn.Module):
    def __init__(self, input_size: int, output_size: int, noise_stddev: float = 0.1):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_size, output_size))
        self.bias = nn.Parameter(torch.zeros(output_size))
        self.noise_stddev = noise_stddev
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x: torch.Tensor, training: bool = False) -> torch.Tensor:
        if training:
            x = x + torch.randn_like(x) * self.noise_stddev
        return x @ self.weight + self.bias

noisy_layer = NoisyLinear(4, 2, noise_stddev=0.2)
base_input = torch.zeros((1, 4))

print('training=True その1:', noisy_layer(base_input, training=True))
print('training=True その2:', noisy_layer(base_input, training=True))
print('training=False   :', noisy_layer(base_input, training=False))

## 回帰プロジェクト: ローカル同梱データで前処理と DNN 回帰を確認する

原本では Auto MPG データを外部 URL から取得していましたが、CI ではネットワーク依存を避けたいので、ここでは scikit-learn に同梱された糖尿病データセットを使います。数値特徴量の標準化、連続値のバケット化、テンソル化、ミニバッチ学習という流れはそのまま確認できます。

In [ ]:
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame.copy()
df.rename(columns={'target': 'disease_progression'}, inplace=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

feature_cols = [col for col in train_df.columns if col != 'disease_progression']
feature_stats = train_df[feature_cols].agg(['mean', 'std']).T

train_features = train_df[feature_cols].copy()
test_features = test_df[feature_cols].copy()

for col in feature_cols:
    mean = feature_stats.loc[col, 'mean']
    std = feature_stats.loc[col, 'std']
    train_features.loc[:, col] = (train_features[col] - mean) / std
    test_features.loc[:, col] = (test_features[col] - mean) / std

bucket_edges = torch.tensor(np.quantile(train_features['bmi'], q=[0.25, 0.5, 0.75]), dtype=torch.float32)
train_bmi_bucket = torch.bucketize(torch.tensor(train_features['bmi'].to_numpy(np.float32)), bucket_edges)
test_bmi_bucket = torch.bucketize(torch.tensor(test_features['bmi'].to_numpy(np.float32)), bucket_edges)

train_bucket_onehot = nn.functional.one_hot(train_bmi_bucket, num_classes=4).float()
test_bucket_onehot = nn.functional.one_hot(test_bmi_bucket, num_classes=4).float()

train_numeric = torch.tensor(train_features.to_numpy(np.float32))
test_numeric = torch.tensor(test_features.to_numpy(np.float32))

x_train_reg = torch.cat([train_numeric, train_bucket_onehot], dim=1)
x_test_reg = torch.cat([test_numeric, test_bucket_onehot], dim=1)
y_train_reg = torch.tensor(train_df['disease_progression'].to_numpy(np.float32))
y_test_reg = torch.tensor(test_df['disease_progression'].to_numpy(np.float32))

reg_loader = DataLoader(
    TensorDataset(x_train_reg, y_train_reg),
    batch_size=32,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

reg_model = nn.Sequential(
    nn.Linear(x_train_reg.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
reg_loss_fn = nn.MSELoss()
reg_optimizer = torch.optim.Adam(reg_model.parameters(), lr=0.01)

reg_history = []
for epoch in range(80):
    reg_model.train()
    epoch_loss = 0.0
    for x_batch, y_batch in reg_loader:
        reg_optimizer.zero_grad()
        preds = reg_model(x_batch).squeeze(1)
        loss = reg_loss_fn(preds, y_batch)
        loss.backward()
        reg_optimizer.step()
        epoch_loss += loss.item() * len(x_batch)
    reg_history.append(epoch_loss / len(reg_loader.dataset))

reg_model.eval()
with torch.no_grad():
    reg_preds = reg_model(x_test_reg).squeeze(1).numpy()

reg_mae = mean_absolute_error(y_test_reg.numpy(), reg_preds)
reg_rmse = root_mean_squared_error(y_test_reg.numpy(), reg_preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(reg_history, color='tab:green')
axes[0].set_title('Regression training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE loss')
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test_reg.numpy(), reg_preds, alpha=0.7)
line_min = min(y_test_reg.min().item(), reg_preds.min())
line_max = max(y_test_reg.max().item(), reg_preds.max())
axes[1].plot([line_min, line_max], [line_min, line_max], '--', color='black')
axes[1].set_title('Predicted vs observed')
axes[1].set_xlabel('Observed')
axes[1].set_ylabel('Predicted')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame({
    '指標': ['Test MAE', 'Test RMSE', '入力次元'],
    '値': [round(reg_mae, 3), round(reg_rmse, 3), int(x_train_reg.shape[1])],
})

## 画像分類プロジェクト: 軽量な digits データセットで MLP を学習する

原本では `torchvision.datasets.MNIST` を使っていましたが、ダウンロードを伴うため CI には不向きです。ここでは同じ 10 クラス分類の流れを、scikit-learn 同梱の `load_digits` に置き換えて再現します。入力は 8×8 の手書き数字画像で、MLP による分類という学習目的は保たれます。

In [ ]:
digits = load_digits()
X = (digits.images / 16.0).astype(np.float32)
y = digits.target.astype(np.int64)

x_train_img, x_temp_img, y_train_img, y_temp_img = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
x_valid_img, x_test_img, y_valid_img, y_test_img = train_test_split(
    x_temp_img, y_temp_img, test_size=0.5, random_state=SEED, stratify=y_temp_img
)

x_train_tensor = torch.tensor(x_train_img.reshape(len(x_train_img), -1))
x_valid_tensor = torch.tensor(x_valid_img.reshape(len(x_valid_img), -1))
x_test_tensor = torch.tensor(x_test_img.reshape(len(x_test_img), -1))
y_train_tensor = torch.tensor(y_train_img)
y_valid_tensor = torch.tensor(y_valid_img)
y_test_tensor = torch.tensor(y_test_img)

img_loader = DataLoader(
    TensorDataset(x_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

classifier = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
)
clf_loss_fn = nn.CrossEntropyLoss()
clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)

train_acc_history = []
valid_acc_history = []
for epoch in range(12):
    classifier.train()
    train_correct = 0
    total = 0
    for x_batch, y_batch in img_loader:
        clf_optimizer.zero_grad()
        logits = classifier(x_batch)
        loss = clf_loss_fn(logits, y_batch)
        loss.backward()
        clf_optimizer.step()
        train_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(x_batch)

    classifier.eval()
    with torch.no_grad():
        valid_logits = classifier(x_valid_tensor)
        valid_preds = valid_logits.argmax(dim=1)
        valid_acc = accuracy_score(y_valid_img, valid_preds.numpy())

    train_acc_history.append(train_correct / total)
    valid_acc_history.append(valid_acc)

classifier.eval()
with torch.no_grad():
    test_logits = classifier(x_test_tensor)
    test_preds = test_logits.argmax(dim=1).numpy()

test_accuracy = accuracy_score(y_test_img, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_acc_history, label='train')
axes[0].plot(valid_acc_history, label='valid')
axes[0].set_title('Digits accuracy curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.0, 1.02)
axes[0].grid(alpha=0.3)
axes[0].legend()

sample_strip = np.hstack([img for img in x_test_img[:8]])
axes[1].imshow(sample_strip, cmap='gray_r')
axes[1].set_title('First 8 test images')
axes[1].axis('off')
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame({
    '項目': ['Validation accuracy (last epoch)', 'Test accuracy'],
    '値': [round(valid_acc_history[-1], 4), round(test_accuracy, 4)],
})

## まとめ

この移行版では、原本 `ch13` の主題である計算グラフ、自動微分、`nn.Module`、カスタムレイヤー、回帰、画像分類を、現行の `torch` と `scikit-learn` だけで継続検証できる形にまとめ直しました。原本サブモジュールのファイルは参照のみで、`machine-learning-book/` 配下は変更していません。